In [7]:
# Importação das bibliotecas necessárias

import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.8f}'.format)

In [8]:
def bissecao(f, a, b, tol=1e-10, max_iter=200):
    """Método da Bisseção"""
    start = time.time()
    if f(a) * f(b) >= 0:
        return None, 0, False, (time.time() - start)

    for i in range(max_iter):
        c = (a + b) / 2
        if abs(f(c)) < tol or (b - a)/2 < tol:
            return c, i+1, True, (time.time() - start)

        if f(a) * f(c) < 0:
            b = c
        else:
            a = c

    return c, max_iter, False, (time.time() - start)

def newton_raphson(f, df, x0, tol=1e-10, max_iter=200):
    """
    Método de Newton-Raphson.
    Requer a função f(x) e sua derivada df(x).
    """
    start = time.time()
    x = x0

    for i in range(max_iter):
        try:
            fx = f(x)
            dfx = df(x)

            if abs(dfx) < 1e-15: # Evita divisão por zero
                return x, i+1, False, (time.time() - start)

            x_new = x - (fx / dfx)

            if abs(x_new - x) < tol or abs(fx) < tol:
                return x_new, i+1, True, (time.time() - start)

            x = x_new
        except (ValueError, OverflowError):
            return x, i+1, False, (time.time() - start)

    return x, max_iter, False, (time.time() - start)

def secante(f, x0, x1, tol=1e-10, max_iter=200):
    """
    Método da Secante.
    Requer dois pontos iniciais x0 e x1.
    """
    start = time.time()

    for i in range(max_iter):
        try:
            fx0 = f(x0)
            fx1 = f(x1)

            if abs(fx1 - fx0) < 1e-15: # Evita divisão por zero
                return x1, i+1, False, (time.time() - start)

            # Fórmula da Secante
            x_new = x1 - (fx1 * (x1 - x0)) / (fx1 - fx0)

            if abs(x_new - x1) < tol or abs(f(x_new)) < tol:
                return x_new, i+1, True, (time.time() - start)

            x0 = x1
            x1 = x_new
        except (ValueError, OverflowError):
            return x1, i+1, False, (time.time() - start)

    return x1, max_iter, False, (time.time() - start)

In [9]:
# Lista contendo dicionários com função, derivada, intervalo e chute inicial
problemas = [
    {
        "id": "a",
        "eq_str": "2x^4 + 4x^3 + 3x^2 - 10x - 15",
        "f": lambda x: 2*x**4 + 4*x**3 + 3*x**2 - 10*x - 15,
        "df": lambda x: 8*x**3 + 12*x**2 + 6*x - 10, # Derivada analítica
        "intervalo": [0, 3],
        "x0_newton": 1.5 # Chute no meio do intervalo
    },
    {
        "id": "b",
        "eq_str": "(x+3)(x+1)(x-2)^3",
        "f": lambda x: (x+3)*(x+1)*(x-2)**3,
        # Derivada usando regra do produto expandida
        "df": lambda x: (2*x + 4)*((x-2)**3) + (x**2 + 4*x + 3)*(3*(x-2)**2),
        "intervalo": [0, 5],
        "x0_newton": 2.5
    },
    {
        "id": "c",
        "eq_str": "5x^3 + x^2 - exp(1-2x) + cos(x) + 20",
        "f": lambda x: 5*x**3 + x**2 - np.exp(1-2*x) + np.cos(x) + 20,
        "df": lambda x: 15*x**2 + 2*x + 2*np.exp(1-2*x) - np.sin(x),
        "intervalo": [-5, 5],
        "x0_newton": 0.5
    },
    {
        "id": "d",
        "eq_str": "sin(x)x + 4",
        "f": lambda x: np.sin(x)*x + 4,
        "df": lambda x: np.cos(x)*x + np.sin(x),
        "intervalo": [1, 5],
        "x0_newton": 4
    },
    {
        "id": "e",
        "eq_str": "(x-3)^5 * ln(x)",
        "f": lambda x: ((x-3)**5) * np.log(x),
        "df": lambda x: 5*((x-3)**4)*np.log(x) + ((x-3)**5)/x,
        "intervalo": [2, 5],
        "x0_newton": 4
    },
    {
        "id": "f",
        "eq_str": "x^10 - 1",
        "f": lambda x: x**10 - 1,
        "df": lambda x: 10*x**9,
        "intervalo": [0.8, 1.2],
        "x0_newton": 1.1
    }
]

In [10]:
resultados = []

for p in problemas:
    f = p['f']
    df = p['df']
    a, b = p['intervalo']
    x0 = p['x0_newton']

    # Método da Bisseção
    r_bis, it_bis, conv_bis, t_bis = bissecao(f, a, b)
    resultados.append({
        "Questão": p['id'],
        "Método": "Bisseção",
        "Raiz": r_bis,
        "Iterações": it_bis,
        "Convergiu": conv_bis,
        "Tempo (s)": t_bis
    })

    # Método de Newton-Raphson
    r_new, it_new, conv_new, t_new = newton_raphson(f, df, x0)
    resultados.append({
        "Questão": p['id'],
        "Método": "Newton",
        "Raiz": r_new,
        "Iterações": it_new,
        "Convergiu": conv_new,
        "Tempo (s)": t_new
    })

    # Método da Secante
    r_sec, it_sec, conv_sec, t_sec = secante(f, a, b)
    resultados.append({
        "Questão": p['id'],
        "Método": "Secante",
        "Raiz": r_sec,
        "Iterações": it_sec,
        "Convergiu": conv_sec,
        "Tempo (s)": t_sec
    })

# Criação do DataFrame para exibição
df_resultados = pd.DataFrame(resultados)

# Exibindo Tabela Formatada
print("Tabela Comparativa de Desempenho")
print("Comparando com Newton e Secante")
display(df_resultados)

# Análise
print("Newton: Geralmente o mais rápido em número de iterações (convergência quadrática).")
print("Secante: Desempenho próximo ao de Newton, mas sem precisar calcular derivadas.")
print("Bisseção: Mais lento, porém robusto (sempre converge se houver troca de sinal).")

Tabela Comparativa de Desempenho
Comparando com Newton e Secante


,Questão,Método,Raiz,Iterações,Convergiu,Tempo (s)
0,a,Bisseção,1.49287871,35,True,0.00008416
1,a,Newton,1.49287871,4,True,0.00000739
2,a,Secante,-1.30038413,10,True,0.00002313
3,b,Bisseção,2.00012207,13,True,0.00001764
4,b,Newton,2.00011324,21,True,0.00002408
5,b,Secante,2.00017819,30,True,0.00003171
6,c,Bisseção,-0.92956046,37,True,0.00040793
7,c,Newton,-0.92956046,10,True,0.00007772
8,c,Secante,-0.92956046,22,True,0.00022316
9,d,Bisseção,4.32323954,34,True,0.00048590


Newton: Geralmente o mais rápido em número de iterações (convergência quadrática).
Secante: Desempenho próximo ao de Newton, mas sem precisar calcular derivadas.
Bisseção: Mais lento, porém robusto (sempre converge se houver troca de sinal).
